# Building the 2009 to 2026 Dataframe

The same idea as `new_features/build_dataframe.ipynb` (one row per 10-K or 10-Q, the label is a bankruptcy 8-K within 365 days), with more of everything:

1. **More years**: the SEC Financial Statement Data Sets from 2009 Q1 to 2026 Q3 (71 quarters), read straight from the zip files in `data/zips/` (`download_zips.sh`).
2. **More numbers**: the 8 variables, the 4 numbers of the 5 ratios, and 25 more from the feature research.
3. **Warning signs from SEC full text search** (`collect_efts.py`, saved in `data/efts/`): going concern doubt and 8 other warning phrases in the filing's text, late filing notices (NT 10-K, NT 10-Q), and 8-Ks about debt defaults (Item 2.04), delisting notices (3.01), restatements (4.02), officers leaving (5.02) and impairments (2.06).
4. **Features**: the 68 from `new_features/feature_research.ipynb`, the warning signs, the company's history (the reports before) and comparisons with the same industry.

Only facts known on the filing date are used for a report's features.

In [1]:
import glob
import json
import os
import time
import zipfile

import numpy as np
import pandas as pd

QUARTERS = [f"{y}q{q}" for y in range(2009, 2027) for q in range(1, 5)][:71]
#the same label date as build_dataframe.ipynb
LABEL_DATE = pd.Timestamp("2026-10-17")


def read_from_zip(quarter, name, **kwargs):
    with zipfile.ZipFile(f"data/zips/{quarter}.zip") as z:
        with z.open(name) as f:
            return pd.read_csv(f, sep="\t", low_memory=False, **kwargs)


print(len(QUARTERS), "quarters:", QUARTERS[0], "to", QUARTERS[-1])

71 quarters: 2009q1 to 2026q3


## One Row per 10-K and 10-Q

From the sub files: 10-K, 10-Q and their amendments. For each company and period I keep the one filed first, since that's the one we would have had at that time (the same as `build_dataframe.ipynb`).

In [2]:
start = time.time()
sub_columns = ["adsh", "cik", "name", "sic", "form", "period", "fy", "fp", "filed", "afs"]
subs = pd.concat([read_from_zip(q, "sub.txt", usecols=sub_columns).assign(quarter=q) for q in QUARTERS])
filings = subs[subs["form"].isin(["10-K", "10-Q", "10-K/A", "10-Q/A"]) & subs["period"].notna()].copy()
filings["ddate"] = filings["period"].astype(int)
#the cik and sic columns can come out as text when a quarter's file is small, so they are made numbers here
filings["cik"] = filings["cik"].astype("int64")
filings["sic"] = pd.to_numeric(filings["sic"], errors="coerce")
filings = filings.sort_values(["cik", "ddate", "filed", "adsh"]).drop_duplicates(["cik", "ddate"]).reset_index(drop=True)

#how many quarters the year so far covers
filings["qtrs_wanted"] = filings["fp"].map({"Q1": 1, "Q2": 2, "Q3": 3, "FY": 4})
filings.loc[filings["form"].str.startswith("10-K"), "qtrs_wanted"] = 4
filings = filings[filings["qtrs_wanted"].notna()]
filings["filed_date"] = pd.to_datetime(filings["filed"].astype(int).astype(str), format="%Y%m%d")
print("filings:", len(filings), " companies:", filings["cik"].nunique(), " took", round(time.time() - start), "s")
print(filings.groupby(filings["filed_date"].dt.year).size())

/var/folders/79/fbgym5f51g38htczmg5vvbh00000gp/T/ipykernel_54069/3102173710.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  subs = pd.concat([read_from_zip(q, "sub.txt", usecols=sub_columns).assign(quarter=q) for q in QUARTERS])


filings: 396458  companies: 15210  took 2 s
filed_date
2009      907
2010     3729
2011    17179
2012    30300
2013    28986
2014    28738
2015    27885
2016    25331
2017    24881
2018    24019
2019    23271
2020    22646
2021    25339
2022    26400
2023    24747
2024    23026
2025    22324
2026    16750
dtype: int64


## The Numbers from the Num Files

Company level totals only (no segments or co-registrants), in US dollars (share counts in shares). Balance sheet tags at qtrs 0, the other tags for the year so far, turned into 12 month numbers. If the first tag is empty I use the next one in the list.

In [3]:
balance_tags = {
    "total_assets": ["Assets"],
    "current_assets": ["AssetsCurrent"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "total_liabilities": ["Liabilities"],
    "liabilities_and_equity": ["LiabilitiesAndStockholdersEquity"],
    "equity_with_minority": ["StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest", "StockholdersEquity"],
    "cash": ["CashAndCashEquivalentsAtCarryingValue", "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents", "Cash"],
    "retained_earnings": ["RetainedEarningsAccumulatedDeficit"],
    "equity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "receivables": ["AccountsReceivableNetCurrent"],
    "inventory": ["InventoryNet"],
    "payables": ["AccountsPayableCurrent", "AccountsPayableAndAccruedLiabilitiesCurrent"],
    "ppe": ["PropertyPlantAndEquipmentNet"],
    "goodwill": ["Goodwill"],
    "intangibles": ["IntangibleAssetsNetExcludingGoodwill", "FiniteLivedIntangibleAssetsNet"],
    "long_term_debt": ["LongTermDebtNoncurrent", "LongTermDebt"],
    "debt_due_in_a_year": ["LongTermDebtCurrent", "DebtCurrent", "NotesPayableCurrent"],
    "lease_liabilities": ["OperatingLeaseLiabilityNoncurrent"],
}
flow_tags = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues",
                "RevenueFromContractWithCustomerIncludingAssessedTax", "SalesRevenueNet"],
    "net_income": ["NetIncomeLoss", "ProfitLoss"],
    "operating_income": ["OperatingIncomeLoss"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities",
                            "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
    "interest_expense": ["InterestExpense", "InterestExpenseNonoperating", "InterestExpenseDebt"],
    "gross_profit": ["GrossProfit"],
    "sga": ["SellingGeneralAndAdministrativeExpense"],
    "rnd": ["ResearchAndDevelopmentExpense"],
    "depreciation": ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization", "Depreciation"],
    "investing_cash_flow": ["NetCashProvidedByUsedInInvestingActivities"],
    "financing_cash_flow": ["NetCashProvidedByUsedInFinancingActivities"],
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "stock_comp": ["ShareBasedCompensation"],
    "stock_issued": ["ProceedsFromIssuanceOfCommonStock"],
    "debt_issued": ["ProceedsFromIssuanceOfLongTermDebt"],
    "debt_repaid": ["RepaymentsOfLongTermDebt"],
    "interest_paid": ["InterestPaidNet", "InterestPaid"],
    "dividends": ["PaymentsOfDividendsCommonStock", "PaymentsOfDividends"],
    "debt_extinguishment": ["GainsLossesOnExtinguishmentOfDebt"],
}
share_tags = {"shares_outstanding": ["CommonStockSharesOutstanding"]}
named = {**balance_tags, **flow_tags, **share_tags}
all_tags = sorted(set(sum(named.values(), [])))
balance_names = set(sum({**balance_tags, **share_tags}.values(), []))
share_names = set(sum(share_tags.values(), []))

start = time.time()
values_list = []
for q in QUARTERS:
    num_q = read_from_zip(q, "num.txt", usecols=["adsh", "tag", "ddate", "qtrs", "uom", "segments", "coreg", "value"])
    num_q = num_q[num_q["tag"].isin(all_tags) & num_q["segments"].isna() & num_q["coreg"].isna() & num_q["value"].notna()]
    num_q = num_q[np.where(num_q["tag"].isin(share_names), num_q["uom"] == "shares", num_q["uom"] == "USD")]
    num_q = num_q.merge(filings[["adsh", "ddate", "qtrs_wanted"]], on=["adsh", "ddate"])
    is_balance = num_q["tag"].isin(balance_names)
    num_q = num_q[(is_balance & (num_q["qtrs"] == 0)) | (~is_balance & (num_q["qtrs"] == num_q["qtrs_wanted"]))]
    values_list.append(num_q[["adsh", "tag", "value"]].drop_duplicates(["adsh", "tag"]))
print("read in", round((time.time() - start) / 60, 1), "minutes")

values_wide = pd.concat(values_list).drop_duplicates(["adsh", "tag"]).pivot(index="adsh", columns="tag", values="value")
for tag in all_tags:
    if tag not in values_wide:
        values_wide[tag] = np.nan
for name, tag_list in named.items():
    column = values_wide[tag_list[0]]
    for backup in tag_list[1:]:
        column = column.fillna(values_wide[backup])
    values_wide[name] = column
#a lot of companies don't use the Liabilities tag, so total liabilities = liabilities and equity - equity
values_wide["total_liabilities"] = values_wide["total_liabilities"].fillna(
    values_wide["liabilities_and_equity"] - values_wide["equity_with_minority"])

numbers = [n for n in named if n not in ("liabilities_and_equity", "equity_with_minority")]
df = filings.merge(values_wide[numbers], left_on="adsh", right_index=True, how="left")
for name in flow_tags:
    df[name] = df[name] * 4 / df["qtrs_wanted"]
#rows with no balance sheet at all can't be scored
df = df[df["total_assets"].notna()].reset_index(drop=True)
print(df.shape)
print(df[numbers].notna().mean().round(3).sort_values(ascending=False))

read in 2.8 minutes


/var/folders/79/fbgym5f51g38htczmg5vvbh00000gp/T/ipykernel_54069/3094126210.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  values_wide = pd.concat(values_list).drop_duplicates(["adsh", "tag"]).pivot(index="adsh", columns="tag", values="value")


(388251, 50)
total_assets           1.000
total_liabilities      0.987
operating_cash_flow    0.977
cash                   0.969
net_income             0.966
equity                 0.958
retained_earnings      0.896
financing_cash_flow    0.814
current_assets         0.777
current_liabilities    0.772
investing_cash_flow    0.743
depreciation           0.743
operating_income       0.740
ppe                    0.730
payables               0.703
shares_outstanding     0.700
revenue                0.690
stock_comp             0.641
capex                  0.579
interest_expense       0.579
interest_paid          0.504
receivables            0.477
intangibles            0.438
goodwill               0.438
inventory              0.397
gross_profit           0.358
sga                    0.317
long_term_debt         0.300
dividends              0.296
debt_due_in_a_year     0.282
rnd                    0.270
stock_issued           0.252
debt_repaid            0.203
lease_liabilities      0.200
d

## Labels

Bankruptcy = 1 if the company filed an 8-K with Item 1.03 (bankruptcy or receivership) in the 365 days after the report, the same rule as `build_dataframe.ipynb`. Only 8-Ks where 1.03 is one of the items count, and every company on the 8-K (a parent and its subsidiaries) gets it. Reports from companies that already had a bankruptcy 8-K on the same day or in the 2 years before are marked, since they are not a prediction.

In [4]:
def read_search(name):
    rows = sum([json.load(open(p)) for p in sorted(glob.glob(f"data/efts/{name}/*.json"))], [])
    events = pd.DataFrame(rows)
    events = events.explode("ciks").rename(columns={"ciks": "cik"})
    events["cik"] = events["cik"].astype(int)
    events["date"] = pd.to_datetime(events["file_date"])
    return events[events["date"] <= LABEL_DATE]


bk = read_search("item_1.03")
bk = bk[bk["items"].apply(lambda items: "1.03" in (items or []))][["cik", "date"]].drop_duplicates()
print("bankruptcy 8-Ks:", len(bk), " companies:", bk["cik"].nunique())
print(bk.groupby(bk["date"].dt.year).size())

pairs = df[["adsh", "cik", "filed_date"]].merge(bk, on="cik")
pairs["days"] = (pairs["date"] - pairs["filed_date"]).dt.days
next_bk = pairs[(pairs["days"] > 0) & (pairs["days"] <= 365)].groupby("adsh")["date"].min()
df["bk_date"] = df["adsh"].map(next_bk)
df["days_to_bankruptcy"] = (df["bk_date"] - df["filed_date"]).dt.days
df["Bankruptcy"] = df["bk_date"].notna().astype(float)
df["already_bankrupt"] = df["adsh"].isin(pairs[(pairs["days"] <= 0) & (pairs["days"] >= -730)]["adsh"])
df["label_known"] = df["filed_date"] + pd.Timedelta(days=365) <= LABEL_DATE
df.loc[~df["label_known"], "Bankruptcy"] = np.nan
ready = df[df["label_known"] & ~df["already_bankrupt"]]
print(ready.groupby(ready["filed_date"].dt.year)["Bankruptcy"].agg(["size", "sum", "mean"]).round(4))

bankruptcy 8-Ks: 2745  companies: 1523
date
2007    150
2008    210
2009    345
2010    199
2011    148
2012    127
2013    115
2014    107
2015    103
2016    172
2017    116
2018     86
2019    115
2020    177
2021     78
2022     42
2023    157
2024    156
2025     87
2026     55
dtype: int64
             size    sum    mean
filed_date                      
2009          874    0.0  0.0000
2010         3643    2.0  0.0005
2011        16542  104.0  0.0063
2012        29062  177.0  0.0061
2013        27949  124.0  0.0044
2014        27863  169.0  0.0061
2015        27039  289.0  0.0107
2016        24599  234.0  0.0095
2017        24131  166.0  0.0069
2018        23353  167.0  0.0072
2019        22741  250.0  0.0110
2020        22128  155.0  0.0070
2021        24791   64.0  0.0026
2022        25925  197.0  0.0076
2023        24394  313.0  0.0128
2024        22738  192.0  0.0084
2025        16635   95.0  0.0057


## Warning Signs

- **In the report itself**: the report's text has "substantial doubt" and "going concern", "forbearance agreement", "minimum bid price" (delisting risk), "chapter 11", "strategic alternatives" (often said before a sale or a bankruptcy), "material weakness" (in internal control), "not in compliance" (often with debt covenants), "reverse stock split" (to keep a listing), "restructuring support agreement", and the stricter or newer "identified a material weakness", "not in compliance with" + "covenants", "notice of default", "debtor-in-possession", "cease operations" and "wind down" (from full text search, so the words can be anywhere in the filing or its exhibits).
- **Before the report**: how many 8-Ks of each warning item and late filing notices the company filed in the 365 days up to the filing date, and the days since the last one (empty if none since 2007). Going concern doubt in the company's earlier reports counts the same way.

In [5]:
event_items = {"item_2.04": "2.04", "item_3.01": "3.01", "item_4.02": "4.02", "item_5.02": "5.02", "item_2.06": "2.06"}
text_searches = ["going_concern", "forbearance", "minimum_bid_price", "chapter_11", "strategic_alternatives",
                 "material_weakness", "not_in_compliance", "reverse_stock_split", "restructuring_support_agreement",
                 "identified_material_weakness", "covenant_not_in_compliance", "notice_of_default", "debtor_in_possession",
                 "cease_operations", "wind_down"]
df = df.sort_values("filed_date").reset_index(drop=True)


def add_event_counts(events, name):
    #events up to the filing date, counted with a running total per company
    events = events[["cik", "date"]].drop_duplicates().sort_values("date")
    events["total"] = events.groupby("cik").cumcount() + 1
    up_to_now = pd.merge_asof(df[["filed_date", "cik"]], events.rename(columns={"date": "last_date"}),
                              left_on="filed_date", right_on="last_date", by="cik", direction="backward")
    year_before = pd.merge_asof(df[["filed_date", "cik"]].assign(t=df["filed_date"] - pd.Timedelta(days=365)), events,
                                left_on="t", right_on="date", by="cik", direction="backward")
    df[f"{name}_last_year"] = up_to_now["total"].fillna(0).values - year_before["total"].fillna(0).values
    df[f"days_since_{name}"] = (df["filed_date"] - up_to_now["last_date"]).dt.days.values


for name, item in event_items.items():
    events = read_search(name)
    add_event_counts(events[events["items"].apply(lambda items: item in (items or []))], name.replace(".", "_"))
add_event_counts(read_search("nt_filing"), "late_filing_notice")

for name in text_searches:
    hits = read_search(name)
    df[f"{name}_in_report"] = df["adsh"].isin(set(hits["adsh"])).astype(float)
    #only the company's 10-K and 10-Q filed before this one (not this one)
    earlier = hits[hits["form"].str.startswith(("10-K", "10-Q"))].copy()
    earlier["date"] = earlier["date"] + pd.Timedelta(days=1)
    add_event_counts(earlier, f"{name}_before")
warning_signs = [c for c in df.columns if c.endswith(("_last_year", "_in_report")) or c.startswith("days_since_")]
ready = df[df["label_known"] & ~df["already_bankrupt"]]
print(pd.DataFrame({"share with a sign": (ready[warning_signs].fillna(0) > 0).mean(),
                    "bankrupt when sign": [ready.loc[ready[c].fillna(0) > 0, "Bankruptcy"].mean() if not c.startswith("days") else np.nan for c in warning_signs],
                    "bankrupt when not": [ready.loc[ready[c].fillna(0) == 0, "Bankruptcy"].mean() if not c.startswith("days") else np.nan for c in warning_signs]}).round(4))

                                                   share with a sign  \
item_2_04_last_year                                           0.0107   
days_since_item_2_04                                          0.0754   
item_3_01_last_year                                           0.0723   
days_since_item_3_01                                          0.2406   
item_4_02_last_year                                           0.0326   
days_since_item_4_02                                          0.1769   
item_5_02_last_year                                           0.7095   
days_since_item_5_02                                          0.9005   
item_2_06_last_year                                           0.0133   
days_since_item_2_06                                          0.1113   
late_filing_notice_last_year                                  0.1980   
days_since_late_filing_notice                                 0.4113   
going_concern_in_report                                       0.

## Features

The 68 candidate features from `new_features/feature_research.ipynb` (the same formulas), then:

- **History**: the change since the company's previous report (any form), how many of the last 4 reports had a loss, how many reports the company has filed since 2009 and the years since its first one (young companies fail more).
- **Industry comparison**: where the company stands among the same industry (2-digit SIC) in the same calendar quarter, as a percentile (like the peer comparison the investigator agent does).

In [6]:
d = df
d["filer_status"] = pd.to_numeric(d["afs"].str[0], errors="coerce")
period_end = pd.to_datetime(d["ddate"].astype(int).astype(str), format="%Y%m%d", errors="coerce")
d["period_end"] = period_end
d["filing_delay_days"] = (d["filed_date"] - period_end).dt.days
is_10k = d["form"].str.startswith("10-K")
deadline = np.where(is_10k, d["filer_status"].map({1: 60, 2: 75}).fillna(90), d["filer_status"].map({1: 40, 2: 40}).fillna(45))
d["late_filer"] = (d["filing_delay_days"] > deadline).astype(float)
d["is_10k"] = is_10k.astype(float)
d["industry"] = (d["sic"] // 100).astype(float)

ta, tl = d["total_assets"], d["total_liabilities"]
working_capital = d["current_assets"] - d["current_liabilities"]
total_debt = d["long_term_debt"].fillna(0) + d["debt_due_in_a_year"].fillna(0)
total_debt[d["long_term_debt"].isna() & d["debt_due_in_a_year"].isna()] = np.nan
monthly_burn = -d["operating_cash_flow"] / 12

f = pd.DataFrame(index=d.index)
#the 5 ratios
f["cash_to_current_liabilities"] = d["cash"] / d["current_liabilities"]
f["months_of_cash"] = np.where(monthly_burn > 0, d["cash"] / monthly_burn, 120)
f["months_of_cash"] = f["months_of_cash"].clip(upper=120).where(d["cash"].notna() & d["operating_cash_flow"].notna())
f["retained_earnings_to_assets"] = d["retained_earnings"] / ta
f["equity_to_assets"] = d["equity"] / ta
f["interest_coverage"] = d["operating_income"] / d["interest_expense"]
#the classic models
f["working_capital_to_assets"] = working_capital / ta
f["ebit_to_assets"] = d["operating_income"] / ta
f["sales_to_assets"] = d["revenue"] / ta
f["equity_to_liabilities"] = d["equity"] / tl
f["log_assets"] = np.log(ta.where(ta > 0))
f["liabilities_to_assets"] = tl / ta
f["current_liabilities_to_current_assets"] = d["current_liabilities"] / d["current_assets"]
f["liabilities_exceed_assets"] = (tl > ta).astype(float).where(tl.notna())
f["net_income_to_assets"] = d["net_income"] / ta
f["cash_flow_to_liabilities"] = d["operating_cash_flow"] / tl
f["current_ratio"] = d["current_assets"] / d["current_liabilities"]
#liquidity and debt
f["cash_to_assets"] = d["cash"] / ta
f["quick_ratio"] = (d["current_assets"] - d["inventory"].fillna(0)) / d["current_liabilities"]
f["debt_due_to_cash"] = d["debt_due_in_a_year"] / d["cash"]
f["debt_to_assets"] = total_debt / ta
f["lease_liabilities_to_assets"] = d["lease_liabilities"] / ta
f["interest_paid_to_cash_flow"] = d["interest_paid"] / d["operating_cash_flow"].abs()
f["ebitda_to_liabilities"] = (d["operating_income"] + d["depreciation"]) / tl
f["payables_to_assets"] = d["payables"] / ta
#profit and costs
f["gross_margin"] = d["gross_profit"] / d["revenue"]
f["operating_margin"] = d["operating_income"] / d["revenue"]
f["net_margin"] = d["net_income"] / d["revenue"]
f["cash_flow_to_assets"] = d["operating_cash_flow"] / ta
f["sga_to_revenue"] = d["sga"] / d["revenue"]
f["rnd_to_assets"] = d["rnd"] / ta
f["stock_comp_to_assets"] = d["stock_comp"] / ta
f["goodwill_to_assets"] = (d["goodwill"].fillna(0) + d["intangibles"].fillna(0)) / ta
f["tangible_equity_to_assets"] = (d["equity"] - d["goodwill"].fillna(0) - d["intangibles"].fillna(0)) / ta
f["receivables_to_revenue"] = d["receivables"] / d["revenue"]
f["capex_to_assets"] = d["capex"] / ta
f["has_revenue"] = (d["revenue"] > 0).astype(float)
#financing
f["financing_cash_flow_to_assets"] = d["financing_cash_flow"] / ta
f["investing_cash_flow_to_assets"] = d["investing_cash_flow"] / ta
f["stock_issued_to_assets"] = d["stock_issued"] / ta
f["debt_issued_to_assets"] = d["debt_issued"] / ta
f["debt_repaid_to_assets"] = d["debt_repaid"] / ta
f["pays_dividends"] = (d["dividends"] > 0).astype(float)
f["debt_extinguishment_to_assets"] = d["debt_extinguishment"] / ta
df = pd.concat([d, f], axis=1)
ratio_features = list(f.columns)
print(len(ratio_features), "ratio features")

43 ratio features


In [7]:
#trends: the same company's report whose period ended about a year before (within 45 days)
history_columns = ["total_assets", "total_liabilities", "revenue", "net_income", "cash", "equity", "operating_cash_flow", "shares_outstanding"]
year_before = df[["cik", "period_end"] + history_columns].rename(columns={c: c + "_year_before" for c in history_columns})
year_before = year_before.rename(columns={"period_end": "period_year_before"}).sort_values("period_year_before")
lookup = df[["adsh", "cik", "period_end"]].assign(target=df["period_end"] - pd.Timedelta(days=365)).sort_values("target")
matched = pd.merge_asof(lookup, year_before, left_on="target", right_on="period_year_before", by="cik",
                        direction="nearest", tolerance=pd.Timedelta(days=45))
matched = matched[matched["period_year_before"] < matched["period_end"]]
df = df.merge(matched.drop(columns=["cik", "period_end", "target"]), on="adsh", how="left")

d, yb = df, lambda c: df[c + "_year_before"]
t = pd.DataFrame(index=d.index)
t["revenue_growth"] = (d["revenue"] - yb("revenue")) / yb("revenue").abs()
t["asset_growth"] = (d["total_assets"] - yb("total_assets")) / yb("total_assets").abs()
t["cash_change_to_assets"] = (d["cash"] - yb("cash")) / yb("total_assets")
t["equity_change_to_assets"] = (d["equity"] - yb("equity")) / yb("total_assets")
t["liabilities_to_assets_change"] = d["total_liabilities"] / d["total_assets"] - yb("total_liabilities") / yb("total_assets")
t["net_income_to_assets_change"] = d["net_income"] / d["total_assets"] - yb("net_income") / yb("total_assets")
t["share_growth"] = (d["shares_outstanding"] - yb("shares_outstanding")) / yb("shares_outstanding")
t["loss_two_years"] = ((d["net_income"] < 0) & (yb("net_income") < 0)).astype(float).where(yb("net_income").notna())
t["net_income_change"] = (d["net_income"] - yb("net_income")) / (d["net_income"].abs() + yb("net_income").abs())

#history: the company's reports before this one, in the order they were filed
df = pd.concat([d, t], axis=1).sort_values(["cik", "filed_date", "adsh"]).reset_index(drop=True)
by_company = df.groupby("cik")
for c in ["cash_to_assets", "net_income_to_assets", "liabilities_to_assets", "cash_flow_to_assets"]:
    df[f"{c}_change_since_last"] = df[c] - by_company[c].shift(1)
loss = (df["net_income"] < 0).astype(float).where(df["net_income"].notna())
df["losses_in_last_4"] = loss.groupby(df["cik"]).transform(lambda x: x.rolling(4, min_periods=1).sum())
df["reports_so_far"] = by_company.cumcount()
df["years_since_first_report"] = (df["filed_date"] - by_company["filed_date"].transform("min")).dt.days / 365.25
df["days_since_last_report"] = (df["filed_date"] - by_company["filed_date"].shift(1)).dt.days

#industry comparison: percentile within the same 2-digit SIC and calendar quarter
df["calendar_quarter"] = df["filed_date"].dt.to_period("Q").astype(str)
peer_columns = ["net_income_to_assets", "liabilities_to_assets", "cash_to_assets", "cash_flow_to_assets", "equity_to_assets",
                "log_assets", "revenue_growth"]
peer_group = df.groupby(["industry", "calendar_quarter"])
for c in peer_columns:
    df[f"{c}_industry_percentile"] = peer_group[c].rank(pct=True)

#the classic scores
df["altman_z2_risk"] = -(6.56 * df["working_capital_to_assets"] + 3.26 * df["retained_earnings_to_assets"]
                         + 6.72 * df["ebit_to_assets"] + 1.05 * df["equity_to_liabilities"])
df["ohlson_o_score"] = (-1.32 - 0.407 * df["log_assets"] + 6.03 * df["liabilities_to_assets"] - 1.43 * df["working_capital_to_assets"]
                        + 0.0757 * df["current_liabilities_to_current_assets"] - 1.72 * df["liabilities_exceed_assets"]
                        - 2.37 * df["net_income_to_assets"] - 1.83 * df["cash_flow_to_liabilities"]
                        + 0.285 * df["loss_two_years"] - 0.521 * df["net_income_change"])
df["zmijewski_score"] = -4.336 - 4.513 * df["net_income_to_assets"] + 5.679 * df["liabilities_to_assets"] + 0.004 * df["current_ratio"]
print(df.shape)

(388251, 198)


In [8]:
eight = ["total_assets", "current_assets", "current_liabilities", "total_liabilities", "revenue", "net_income",
         "operating_income", "operating_cash_flow"]
groups = {
    "8 variables": eight,
    "accounting ratios": ratio_features + ["altman_z2_risk", "ohlson_o_score", "zmijewski_score"],
    "trends and history": list(t.columns) + [c for c in df.columns if c.endswith("_change_since_last")]
                          + ["losses_in_last_4", "reports_so_far", "years_since_first_report", "days_since_last_report"],
    "industry comparison": [f"{c}_industry_percentile" for c in peer_columns],
    "filing behavior": ["filer_status", "filing_delay_days", "late_filer", "is_10k", "industry"],
    "warning signs": warning_signs,
}
all_features = sum(groups.values(), [])
df[all_features] = df[all_features].replace([np.inf, -np.inf], np.nan).astype("float32")
json.dump(groups, open("feature_groups.json", "w"), indent=1)

keep = ["adsh", "cik", "name", "sic", "form", "fp", "ddate", "filed", "filed_date", "bk_date", "days_to_bankruptcy",
        "Bankruptcy", "already_bankrupt", "label_known"] + numbers + all_features
df = df[list(dict.fromkeys(keep))].sort_values(["filed_date", "adsh"]).reset_index(drop=True)
df.to_pickle("long_dataframe_2009_to_2026.pkl")
print(df.shape, " features:", len(all_features))
print({g: len(fs) for g, fs in groups.items()})

(388251, 182)  features: 140
{'8 variables': 8, 'accounting ratios': 46, 'trends and history': 17, 'industry comparison': 7, 'filing behavior': 5, 'warning signs': 57}
